# MetabTravLR — submit SpaceTravLR runs to SLURM

One cell per dataset. Running a cell submits **one SLURM job** that does setup, training and
artifacts, and returns immediately with the job id. Nothing heavy happens in this notebook.

What the job does (`run_spacetravlr.py`):

| stage | writes |
|---|---|
| `setup` | `spacetravlr_output/input_data/` (processed adata, CellOracle links, NicheNet links) |
| `fit` | `spacetravlr_output/betadata/<gene>_betadata.parquet` |
| `artifacts` | `easy_download/metabtravlr_outputs/<tier>/{gene_pairs.csv, histograms.csv, histograms.png}` and `spacetravlr_adata.h5ad` |

**One job, many cores.** Setup is CPU work — imputation is `magic`, the GRN is sklearn
ridge — but it still needs a GPU node, because importing the package pulls in a CUDA-only
torch. So it cannot be split onto a CPU partition, and the memory has to come from cores
instead: `process_adata_` and CellOracle each copy the whole AnnData, so Human_Lung
(278k × 5k, ~5.6 GB per dense copy) peaks near 50 GB and OOM'd an 8-core allocation. The
default is now **32 cores ≈ 256 GB** on `savio3_gpu`.

**Logs** go to `{METAB_DATA_DIR}/spacetravlr_logs/<DATASET>/<stages>_<timestamp>.log` — a
sibling of `harreman_logs/`, deliberately *outside* `spacetravlr_output/`. SLURM opens
that file before the job body runs, so it cannot live in the directory the job is about
to create. The log now prints a timestamped line per phase.

**Reruns are cheap and safe.** Setup is skipped when it's already complete, and `fit` skips
any gene that already has a betadata parquet — so re-submitting after a timeout resumes
rather than starting over.

Per-dataset settings (cell-type column, tiers, target genes, SLURM resources) live in
`dataset_configs.py` — see the `DATASETS` dict there. Target genes are shared across
datasets by default (`metab_travlr_config.FOCUS_GENES`); metabolite pairs always come from
each dataset's own harreman `metabolite_selection.yaml`.

In [1]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine: walk up from the
# working dir until we hit a repo marker, then put that dir on sys.path.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from metab_processing.SpaceTravLR.submit_spacetravlr import submit
from metab_processing.SpaceTravLR.dataset_configs import DATASETS, get_config

print('datasets:', sorted(DATASETS))
print('target genes:', get_config(sorted(DATASETS)[0])['focus_genes'])

datasets: ['Human_Lung', 'Human_Prostate_Adenocarcinoma', 'Primary_Dermal_Melanoma']
target genes: ['HIF1A', 'TCF7', 'CD3E', 'HAVCR2', 'TOX', 'GAPDH', 'FOXP3', 'CD3D', 'TBP', 'B2M', 'PDCD1', 'LAG3', 'IL2RA', 'ENTPD1', 'ACTB', 'CD4', 'CD3G', 'MYC', 'IL10', 'CTLA4']


## Primary_Dermal_Melanoma

In [ ]:
submit('Primary_Dermal_Melanoma', overwrite=True, cpus_per_task=8)

## Human_Lung

In [9]:
submit('Human_Lung', overwrite=True, cpus_per_task=16)

dataset : Human_Lung
stages  : all (overwrite)
log     : /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/Human_Lung/all_20260803_135159.log
sbatch  : {'account': 'fc_wagnerlabfca', 'partition': 'savio3_gpu', 'qos': 'a40_gpu3_normal', 'cpus_per_task': 16, 'ignore_pbs': True, 'job_name': 'MetabTravLR_Human_Lung', 'output': '/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/Human_Lung/all_20260803_135159.log', 'time': datetime.timedelta(days=1), 'gres': 'gpu:A40:1'}
command : /global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python /global/home/users/fosterangus/Projects/MetabTravLR/SpaceTravLR/metab_processing/SpaceTravLR/run_spacetravlr.py --dataset Human_Lung --overwrite
Submitted batch job 36303679

submitted job 36303679


36303679

In [2]:
submit('Human_Prostate_Adenocarcinoma', overwrite=True, cpus_per_task=16)

dataset : Human_Prostate_Adenocarcinoma
stages  : all (overwrite)
log     : /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/Human_Prostate_Adenocarcinoma/all_20260803_135710.log
sbatch  : {'account': 'fc_wagnerlabfca', 'partition': 'savio3_gpu', 'qos': 'a40_gpu3_normal', 'cpus_per_task': 16, 'ignore_pbs': True, 'job_name': 'MetabTravLR_Human_Prostate_Adenocarcinoma', 'output': '/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/Human_Prostate_Adenocarcinoma/all_20260803_135710.log', 'time': datetime.timedelta(days=1), 'gres': 'gpu:A40:1'}
command : /global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python /global/home/users/fosterangus/Projects/MetabTravLR/SpaceTravLR/metab_processing/SpaceTravLR/run_spacetravlr.py --dataset Human_Prostate_Adenocarcinoma --overwrite
Submitted batch job 36303765

submitted job 36303765


36303765

## Variations

`submit()` takes:

- `stages=['setup'|'fit'|'artifacts']` — run only part of the pipeline, e.g. re-do the
  read-out without retraining.
- `overwrite=True` — delete `input_data/` and redo setup. Needs the `setup` stage (it is
  rejected rather than silently ignored otherwise). **Trained betadata is kept**, so those
  betas came from the *previous* preprocessing — the job log says so.
- `clear_betadata=True` — delete `betadata/`, forcing every gene to retrain. Independent of
  `overwrite`; combine the two for a genuinely clean slate.
- `dry_run=True` — print the sbatch settings and command without submitting.
- any SLURM key as a keyword (`time_hours`, `partition`, `qos`, `gres`, `cpus_per_task`,
  `account`, `python_path`) to override the config for this submission only.

### Customising a dataset

One-off, at the call site:

```python
submit('Human_Lung', time_hours=36, cpus_per_task=16)
```

Permanently, in the `DATASETS` dict in `dataset_configs.py` — any key of `DEFAULTS`, with
`slurm` merging key-by-key so you only name what differs:

```python
DATASETS = {
    'Primary_Dermal_Melanoma': {},
    'Human_Lung': {
        'cell_type_src': 'leiden_scVI_res_1',   # 25 clusters instead of 16
        'beta_group': 'metab',                  # smaller spacetravlr_adata.h5ad
        'slurm': {'time_hours': 36},            # everything else stays default
    },
}
```

A typo'd key raises instead of being silently ignored.

Two jobs may `fit` the same dataset concurrently (the gene queue is lock-based), but **not
`setup`** — a second setup refuses via a `.setup.lock` in `spacetravlr_output/`. A lock left
behind by a killed job is detected (its job id is no longer in `squeue`) and cleared
automatically.

In [6]:
# See exactly what would be submitted, without submitting.
submit('Human_Lung', dry_run=True, cpus_per_task=16)

dataset : Human_Lung
stages  : all
log     : /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/Human_Lung/all_20260802_091916.log
sbatch  : {'account': 'fc_wagnerlabfca', 'partition': 'savio3_gpu', 'qos': 'a40_gpu3_normal', 'cpus_per_task': 16, 'ignore_pbs': True, 'job_name': 'MetabTravLR_Human_Lung', 'output': '/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/Human_Lung/all_20260802_091916.log', 'time': datetime.timedelta(days=1), 'gres': 'gpu:A40:1'}
command : /global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python /global/home/users/fosterangus/Projects/MetabTravLR/SpaceTravLR/metab_processing/SpaceTravLR/run_spacetravlr.py --dataset Human_Lung
dry run -- not submitted


In [ ]:
# Re-do just the read-out after a finished training run (minutes, not hours).
# submit('Primary_Dermal_Melanoma', stages=['artifacts'], time_hours=2)

# Redo setup from scratch and retrain every gene.
# submit('Human_Lung', overwrite=True, clear_betadata=True)

# More memory: it scales with cores (~8 GB each on savio3_gpu), up to 32.
# submit('Human_Lung', cpus_per_task=32, time_hours=36)

# Spawn a second worker on a dataset already training -- the gene queue is lock-based,
# so extra workers just pick up untrained genes.
# submit('Human_Lung', stages=['fit'])

## Monitor

In [6]:
!squeue -u $USER

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          36303679 savio3_gp MetabTra fosteran PD       0:00      1 (Resources)
          36303765 savio3_gp MetabTra fosteran PD       0:00      1 (None)
          36303599 savio3_ht OOD_VSCo fosteran  R       9:34      1 n0183.savio3


In [8]:
# Tail the newest log for a dataset.
from metab_processing.SpaceTravLR.dataset_configs import dataset_paths

# DATASET = 'Primary_Dermal_Melanoma'
DATASET = 'Human_Lung'
# DATASET = 'Human_Prostate_Adenocarcinoma'
logs = sorted(dataset_paths(DATASET)['log_dir'].glob('*.log'))
print(logs[-1] if logs else 'no logs yet')
if logs:
    print(''.join(logs[-1].read_text().splitlines(keepends=True)[-40:]))

/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/Human_Lung/setup_20260731_140613.log
[14:08:36] === Human_Lung | stages=['setup'] | overwrite=True ===
[14:08:36] outdir: /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/spacetravlr_output
[14:08:36] XDG_CACHE_HOME=/tmp/spacetravlr_cache_36172404
[14:08:36] reading /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/adata.h5ad
[14:08:40] adata: 278328 cells x 5001 genes
[14:08:40] setup_ (run_commot=False) ...
[2026-07-31T20:08:16.094] error: *** JOB 36172404 ON n0163.savio3 CANCELLED AT 2026-07-31T20:08:16 DUE TO TIME LIMIT ***



In [6]:
# What has finished training so far.
for dataset in sorted(DATASETS):
    paths = dataset_paths(dataset)
    done = sorted(p.name[:-len('_betadata.parquet')]
                  for p in paths['betadata'].glob('*_betadata.parquet')) \
        if paths['betadata'].exists() else []
    print(f"{dataset:28s} setup={paths['input_data'].is_dir()}  "
          f"beta_adata={paths['beta_adata'].exists()}  genes={done}")

Human_Lung                   setup=True  beta_adata=False  genes=[]
Primary_Dermal_Melanoma      setup=True  beta_adata=True  genes=['CD3E', 'CD3G', 'CD4', 'CTLA4', 'ENTPD1', 'FOXP3', 'HAVCR2', 'HIF1A', 'IL10', 'IL2RA', 'LAG3', 'MYC', 'PDCD1', 'TBP', 'TCF7', 'TOX']


## Publish

Copy each dataset's `easy_download/` (now including `metabtravlr_outputs/`) into the
aggregate `Results/` tree — the same step `quick_start_metab.ipynb` and `run_full_harr.ipynb`
end with. Deliberately **not** part of the job: it touches every dataset, not just the one
that ran, so it's a manual step once the runs you care about have finished.

In [15]:
sys.path.insert(0, str(_root / 'metab_processing'))
from Harreman.copy_easy_download import save_easy_downloads

save_easy_downloads()